# Running EMMS on KIRC when some modalities are missing

Same pipeline as scripts/run.py, just for KIRC and kept in one notebook as to be easy to step through. During training we drop 30% of the WSI and 30% of the RNA on paired patients; at test time every patient still has both. align_weight=0.01, 5 folds, and we sweep lambda over [0,1]. All preprocessing (scalers, k-means) is fit on the training split only.

## Imports and paths

In [11]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import sys, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

warnings.filterwarnings(
    "ignore",
    message=r"std\(\): degrees of freedom is <= 0",
    category=UserWarning,
)

def find_emms_root(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / "EMMS"]
    for candidate in candidates:
        if (candidate / "models" / "evidential_survival.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the EMMS project root")

EMMS_ROOT = find_emms_root()
assert (EMMS_ROOT / "models" / "evidential_survival.py").exists(), f"EMMS not found at {EMMS_ROOT}"

sys.path.insert(0, str(EMMS_ROOT / "models"))
sys.path.insert(0, str(EMMS_ROOT / "configs"))

from evidential_survival import (ENNreg_init, ENN_survival_prediction,
                                 Loss_function, Eval_Loss_function, evreg_evaluation)
from data_loader import DataPreprocessor, MultiModalDataset, get_collate_fn
from trainer import EVREGTrainer
from default_config import get_wsi_rna_with_missing_config


def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_default_dtype(torch.float64)


## Settings for this run

In [12]:
CANCER = 'KIRC'
MISSING_CONFIG = 'WSI:0.3_RNA:0.3'   # only applied to the train split
TEST_SCENARIO = 'Complete'
RNA_GAMMA_SCALE = 0.3
ALIGN_WEIGHT = 0.01                  # KL alignment on complete cases, set 0 to turn off
MODALITIES = ['RNA', 'WSI']

config = get_wsi_rna_with_missing_config(MISSING_CONFIG)
set_seed(config.data.seed)

print('cancer:', CANCER, '| missing(train):', MISSING_CONFIG, '| test:', TEST_SCENARIO)
print('align_weight:', ALIGN_WEIGHT, '| K:', config.model.K)
print('lr/epochs:', config.training.learning_rate, '/', config.training.max_epochs)
print('eval lambdas:', config.eval.eval_lambdas)

cancer: KIRC | missing(train): WSI:0.3_RNA:0.3 | test: Complete
align_weight: 0.01 | K: {'RNA': 50, 'WSI': 50}
lr/epochs: 0.011 / 70
eval lambdas: [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]


## Load the data

In [13]:
preprocessor = DataPreprocessor(config)
preprocessor.load_wsi_embeddings(config.data.titan_embeddings_dir)

  Loading 9547 patient-level WSI embeddings...
  Loaded 9547 patients, embedding dim = 768


## Train and evaluate

One model per fold, trained with the missing pattern on the train split, then scored on the held-out patients with both modalities present.

In [14]:
all_results = []
ckpt_dir = EMMS_ROOT / "results" / "_notebook_kirc_w30r30"
ckpt_dir.mkdir(parents=True, exist_ok=True)

for k_fold in range(5):
    print(f"\n=== fold {k_fold} ===")
    set_seed(config.data.seed)   # reset so folds only differ by the split

    rna_dir   = os.path.join(config.data.mmp_root, 'data_csvs', 'rna', 'hallmarks', CANCER)
    split_dir = os.path.join(config.data.mmp_root, 'splits', 'survival',
                             f'TCGA_{CANCER}_overall_survival_k={k_fold}')
    rna_data    = preprocessor.load_rna_data(rna_dir)
    train_split = pd.read_csv(os.path.join(split_dir, 'train.csv'))
    test_split  = pd.read_csv(os.path.join(split_dir, 'test.csv'))

    # missing only goes on the train split
    result = preprocessor.prepare_data(
        train_split, test_split, modalities=MODALITIES, rna_data=rna_data,
        with_validation=False, random_state=1,
        missing_config_train=config.missing.missing_config_train,
        missing_seed=config.missing.missing_seed,
        missing_complete_cases_only=config.missing.complete_cases_only,
        missing_disjoint=config.missing.disjoint, missing_verbose=True,
    )
    train_data, test_data, rna_cols = result['train'], result['test'], result['rna_cols']

    features_tr, masks_tr, dur_tr, evt_tr = preprocessor.extract_features(train_data, MODALITIES, rna_cols)
    features_te, masks_te, dur_te, evt_te = preprocessor.extract_features(test_data,  MODALITIES, rna_cols)

    # scalers fit on train
    features_tr, dur_tr_t, idx_tr = preprocessor.fit_scalers(features_tr, dur_tr, masks=masks_tr)
    dur_tr_orig = dur_tr[idx_tr]; evt_tr = evt_tr[idx_tr]
    masks_tr = {k: v[idx_tr] for k, v in masks_tr.items()}

    features_te, dur_te_t, idx_te = preprocessor.transform_data(features_te, dur_te, masks=masks_te)
    dur_te_orig = dur_te[idx_te]; evt_te = evt_te[idx_te]
    masks_te = {k: v[idx_te] for k, v in masks_te.items()}

    train_ds   = MultiModalDataset(features_tr, dur_tr_t, evt_tr, masks=masks_tr)
    collate_fn = get_collate_fn(MODALITIES, with_mask=True)
    train_loader = DataLoader(train_ds, batch_size=config.training.batch_size,
                              shuffle=True, collate_fn=collate_fn, num_workers=0)

    # kmeans prototypes from train
    x_init    = {k: torch.tensor(v, dtype=torch.float64) for k, v in features_tr.items()}
    mask_init = {k: torch.tensor(v, dtype=torch.float64) for k, v in masks_tr.items()}
    dur_init  = torch.tensor(dur_tr_t, dtype=torch.float64)
    prototypes, k_dict = ENNreg_init(x_init, dur_init, K_dict=config.model.K,
                                     mask_dict=mask_init, rna_gamma_scale=RNA_GAMMA_SCALE)
    model = ENN_survival_prediction(x_init, k_dict, prototypes)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.training.learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=config.training.milestones,
                                                     gamma=config.training.gamma)
    criterion      = Loss_function(align_weight=ALIGN_WEIGHT)
    criterion_eval = Eval_Loss_function()

    trainer   = EVREGTrainer(model, optimizer, scheduler, criterion, criterion_eval, config)
    save_path = str(ckpt_dir / f"model_k{k_fold}.pth")
    trainer.train_no_val(train_loader, save_path, use_mask=True,
                         print_every=10, max_epochs=config.training.max_epochs, print_hx_every=None)

    # eval on complete (both masks = 1)
    try:
        model.load_state_dict(torch.load(save_path, weights_only=True))
    except TypeError:
        model.load_state_dict(torch.load(save_path))
    model.eval()

    x_test = {k: torch.tensor(v, dtype=torch.float64) for k, v in features_te.items()}
    n_test = len(features_te['RNA'])
    masks_complete = {'RNA': torch.ones(n_test, dtype=torch.float64),
                      'WSI': torch.ones(n_test, dtype=torch.float64)}
    dur_test_t = torch.tensor(dur_te_orig, dtype=torch.float64)

    with torch.no_grad():
        pred = model(x_test, masks=masks_complete)

    for lam in config.eval.eval_lambdas:
        ci, ibs, nbll = evreg_evaluation(pred, dur_test_t, evt_te, weight=lam,
                                         pt=preprocessor.pt, YJ=True,
                                         durations_train=dur_tr_orig, events_train=evt_tr, ibs_bins=4)
        all_results.append({'fold': k_fold, 'lambda': lam,
                            'C-index': ci, 'IBS': ibs, 'NBLL': nbll})

    best = max(r['C-index'] for r in all_results if r['fold'] == k_fold)
    print(f"fold {k_fold} best C-index: {best:.4f}")

df_all = pd.DataFrame(all_results)
print("done,", len(df_all), "rows")
df_all.head()


=== fold 0 ===
Using patient-level WSI embeddings...
  Train: 283 patients, Test: 57 patients
[apply_missing_config] Applied missing_config_train: {'WSI': 0.3, 'RNA': 0.3} seed= 1
  - WSI: valid=0.700, missing=0.300 (on complete-pair candidates)
  - RNA: valid=0.700, missing=0.300 (on complete-pair candidates)
  INFO: [Corr] Applied correlation normalization to 'RNA'
  INFO: [L2] Gamma scaled by c*0.3=0.30 for 'RNA', range: [0.0412, 0.3000]
  INFO: [Corr] Applied correlation normalization to 'WSI'
  INFO: [L2] Gamma scaled by c*0.3=0.30 for 'WSI', range: [0.0448, 0.3000]
  INFO: Added learnable dim_scales for 2 modalities: [4168, 768]
  INFO: Discount init for 'RNA': 0.0 (sigmoid=0.5000)
  INFO: Discount init for 'WSI': 0.0 (sigmoid=0.5000)

Training (NO validation, fixed 70 epochs)...
  Epoch  10 | Train: 8.7279
  Epoch  20 | Train: 4.4168
  Epoch  30 | Train: 3.0344
  Epoch  40 | Train: 2.6267
  Epoch  50 | Train: 1.9621
  Epoch  60 | Train: 2.3114
  Epoch  70 | Train: 1.8698
  -> F

,fold,lambda,C-index,IBS,NBLL
0,0,0.0,0.684959,0.171872,2.682170
1,0,0.1,0.684959,0.154321,1.413792
2,0,0.2,0.686992,0.139994,1.180620
3,0,0.3,0.686992,0.128891,1.039915
4,0,0.4,0.686992,0.121013,0.941197


## Results

In [15]:
summary = (df_all.groupby('lambda')
           .agg(**{'C-index_mean': ('C-index', 'mean'), 'C-index_std': ('C-index', 'std'),
                   'IBS_mean':     ('IBS', 'mean'),     'IBS_std':     ('IBS', 'std'),
                   'NBLL_mean':    ('NBLL', 'mean'),    'NBLL_std':    ('NBLL', 'std')})
           .reset_index())

out_dir = EMMS_ROOT / "results" / "missing_W3_R3" / CANCER
out_dir.mkdir(parents=True, exist_ok=True)
df_all.to_csv(out_dir / "detailed_results_complete.csv", index=False)
summary.to_csv(out_dir / "summary_results_complete.csv", index=False)

print(f"KIRC, missing {MISSING_CONFIG}, complete test, 5-fold\n")
print(summary.round(4).to_string(index=False))

row = summary[np.isclose(summary['lambda'], 1.0)].iloc[0]
print(f"\nlambda=1.0: C-index = {row['C-index_mean']:.4f} +/- {row['C-index_std']:.4f}"
      f"  |  IBS = {row['IBS_mean']:.4f}  |  NBLL = {row['NBLL_mean']:.4f}")
summary

KIRC, missing WSI:0.3_RNA:0.3, complete test, 5-fold

 lambda  C-index_mean  C-index_std  IBS_mean  IBS_std  NBLL_mean  NBLL_std
    0.0        0.7640       0.0897    0.1263   0.0387     2.2400    0.8107
    0.1        0.7657       0.0876    0.1142   0.0350     1.1698    0.2417
    0.2        0.7661       0.0872    0.1042   0.0322     0.9619    0.2088
    0.3        0.7661       0.0872    0.0964   0.0302     0.8312    0.1927
    0.4        0.7661       0.0872    0.0906   0.0291     0.7366    0.1830
    0.5        0.7661       0.0872    0.0870   0.0287     0.6641    0.1769
    0.6        0.7661       0.0872    0.0854   0.0291     0.6076    0.1730
    0.7        0.7661       0.0872    0.0860   0.0301     0.5644    0.1708
    0.8        0.7669       0.0863    0.0887   0.0317     0.5345    0.1703
    0.9        0.7668       0.0861    0.0935   0.0339     0.5228    0.1722
    1.0        0.7625       0.0730    0.1005   0.0366     0.6508    0.1971

lambda=1.0: C-index = 0.7625 +/- 0.0730  |  I

,lambda,C-index_mean,C-index_std,IBS_mean,IBS_std,NBLL_mean,NBLL_std
0,0.0,0.764019,0.089699,0.126310,0.038657,2.239965,0.810694
1,0.1,0.765728,0.087641,0.114213,0.035022,1.169828,0.241683
2,0.2,0.766135,0.087176,0.104230,0.032210,0.961860,0.208792
3,0.3,0.766135,0.087176,0.096360,0.030233,0.831157,0.192665
4,0.4,0.766135,0.087176,0.090604,0.029083,0.736566,0.183033
5,0.5,0.766135,0.087176,0.086962,0.028721,0.664126,0.176880
6,0.6,0.766135,0.087176,0.085433,0.029086,0.607648,0.172982
7,0.7,0.766135,0.087176,0.086018,0.030108,0.564445,0.170811
8,0.8,0.766948,0.086268,0.088717,0.031723,0.534485,0.170294
9,0.9,0.766791,0.086144,0.093529,0.033885,0.522771,0.172163
